| Вариант  | ПУНКТ 2 | ПУНКТ 3 | ПУНКТ 4 | ПУНКТ 6 | ПУНКТ 8 | ПУНКТ 9 | ПУНКТ 10 |
|----------|---------|---------|---------|---------|---------|---------|----------|
| 7 | Apriori, 5%  | ice_crea | 40% | Closeness | PCA | tSNE | 6 |

Объектом анализа является транзакция.
Признаки:
1. Customer — кто покупает
2. Product — что покупают
3. Time — когда покупают

In [1]:
import pandas as pd

Загрузка файла TRANSACTION.csv

In [2]:
csv = "TRANSACTION.csv"
df = pd.read_csv(csv)

Посмотрим наименование столбцов

In [3]:
print(df.columns)

Index(['CUSTOMER', 'TIME', 'PRODUCT'], dtype='str')


In [4]:
col_name_product = "PRODUCT"
col_name_time = "TIME"
col_name_customer = "CUSTOMER"

Количество уникальных значений Product

In [5]:
unumber_product = df[col_name_product].nunique()
print(f"Количество уникальных значений {col_name_product}: {unumber_product}")

unumber_customer = df[col_name_customer].nunique()
print(f"Количество уникальных значений {col_name_customer}: {unumber_customer}")

Количество уникальных значений PRODUCT: 20
Количество уникальных значений CUSTOMER: 1001


Частые эпизоды с ограничением на размер правила равным 4, с использованием
алгоритма и порога на поддержку согласно вашему варианту. 

Подготовим данные для алгоритма Apriori

Сгруппируем в транзакции, которые будут содержать товары

In [6]:
transactions = []
for transaction, group in df.groupby([col_name_customer])[col_name_product]:
    basket = group.tolist()
    transactions.append(basket)

Алгоритм Apriori

In [7]:
limit = 0.05

1. Частоты. Подсчитаем сколько раз каждый товар встречается в корзинах

In [8]:
products = df[col_name_product].unique()
counts = {product: 0 for product in products}
for basket in transactions:
    for item in basket:
        counts[item] = counts[item] + 1

2. Поддержки. support = count / число транзакций. Оставить всё что больше порога (5%).

In [9]:
count_transactions = len(transactions)
L1 = {} # частые одиночные товары

for item, count in counts.items():
    support = count / count_transactions
    if support >= limit:
        L1[item] = support
print(len(list(L1)))

20


3. Строим кандидатов размером 2

In [10]:
from itertools import combinations

products = list(L1.keys())
pairs = list(combinations(products, 2))

4. Построение L2. Проверяем есть ли 2 элемента в корзине. Частоты. Вычисляем support. Фильтруем.

In [11]:
counts = {pair: 0 for pair in pairs}
for basket in transactions:
    for pair in pairs:
        if pair[0] in basket and pair[1] in basket:
            counts[pair] = counts[pair] + 1

L2 = {}
for item, count in counts.items():
    support = count / count_transactions
    if support >= limit:
        L2[item] = support
print(len(list(L2)))

159


Построение множества из 3 уникальных пар

In [15]:
C3 = set()

for p1 in L2:
    for p2 in L2:
        union = set(p1).union(set(p2))
        candidate = tuple(sorted(union))
        if len(union) == 3:
            C3.add(candidate)